## Multimodal Neural Network Approach - Data Preprocessing


In [ ]:
import numpy as np
import os
import pandas as pd
import pickle
import pylab as plt
import seaborn as sns

from imblearn.over_sampling import RandomOverSampler
from skimage.transform import resize
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder

%matplotlib inline

### Data Analysis
Demographic splits after data cleaning, removal of duplicates, and data preprocessing


| Encoded | Labels | Count | Total |                            
| ----------- | ----------- | ----------- |----------- |          
| 0 | AD | 188 | - |                                    
| 1 | MCI | 401 | - |                                    
| 2 | NC | 229 | 818 |

| Sex | Count | Total | 
| ----------- | ----------- | ----------- |
| F | 342 | - |
| M | 476 | 818 |

| Age | Count | Total | 
| ----------- | ----------- | ----------- |
| 55-60 | 25 | - |
| 61-70 | 137 | - |
| 71-80 | 469 | - |
| 81-90 | 186 | - |
| 90+ | 1 | 818 |

### Biomarker Dataset

#### (contains records for 419 patients in ADNI1)

The biomarker dataset consists of 4 features (**CTWHITE, CTRED, PROTEIN, GLUCOSE**) taken at 5 separate time steps for each patient in ADNI1 (**Baseline, M12, M24, M36, M48**), resulting in 20 collected samples for each patient. The resulting Excel table looks like this:

| Patient ID | CTWHITE | CTRED | PROTEIN | GLUCOSE | CTWHITE_M12 | CTRED_M12 | PROTEIN_M12 | GLUCOSE_M12 | CTWHITE_M24 | CTRED_M24 | PROTEIN_M24 | GLUCOSE_M24 | CTWHITE_M36 | CTRED_M36 | PROTEIN_M36 | GLUCOSE_M36 | CTWHITE_M48 | CTRED_M48 | PROTEIN_M48 | GLUCOSE_M48 |
|------------|---------|-------|---------|---------|-------------|-----------|-------------|-------------|-------------|-----------|-------------|-------------|-------------|-----------|-------------|-------------|-------------|-----------|-------------|-------------|
| 002_S_0295 | 0.0     | 9.0   | 48.0    | 65.0    | 0.0         | 14.0      | 49.0        | 65.0        | NaN         | NaN       | NaN         | NaN         | 1.0         | 1.0       | 50.0        | 73.0        | 2.0         | 1.0       | 56.0        | 79.0        |
| 002_S_0413 | 0.0     | 50.0  | 44.0    | 54.0    | 1.0         | 9.0       | 46.0        | 50.0        | NaN         | NaN       | NaN         | NaN         | NaN         | NaN       | NaN         | NaN         | NaN         | NaN       | NaN         | NaN         |
| 002_S_0559 | 2.0     | 0.0   | 37.0    | 47.0    | 1.0         | 1.0       | 44.0        | 59.0        | 1.0         | 1.0       | 40.0        | 65.0        | 1.0         | 13.0      | 44.0        | 62.0        | NaN         | NaN       | NaN         | NaN         |
| 002_S_0619 | 0.0     | 1.0   | 35.0    | 53.0    | 0.0         | 14.0      | 37.0        | 58.0        | NaN         | NaN       | NaN         | NaN         | NaN         | NaN       | NaN         | NaN         | NaN         | NaN       | NaN         | NaN         |
| 002_S_0685 | 0.0     | 2.0   | 53.0    | 43.0    | 1.0         | 13.0      | 46.0        | 50.0        | NaN         | NaN       | NaN         | NaN         | 1.0         | 2.0       | 52.0        | 53.0        | NaN         | NaN       | NaN         | NaN         |
| ...        | ...     | ...   | ...     | ...     | ...         | ...       | ...         | ...         | ...         | ...       | ...         | ...         | ...         | ...       | ...         | ...         | ...         | ...       | ...         | ...         |

This table represents the DataFrame with the specified columns and data values.

To preprocess this, the code begins by identifying all Excel files within the specified directory. It then initialises an empty DataFrame to aggregate the merged data. The function iterates through each Excel file, reading its contents into a temp DataFrame. 

For each file, it extracts the filename without the extension to use as a suffix for column names to differentiate between timesteps. Using an outer join operation, the function merges the current DataFrame (`df`) with the accumulated dataset (`biomarker_dataset`) based on the shared PTID column. 

Then the 'PTID' column is renamed to 'Patient ID' for consistency. The merged DataFrame is then printed, and the count of NaN values in each column is computed and displayed. Finally, the consolidated dataset is saved as a CSV file (BiomarkerDataCSV.csv) without including the index.

In [ ]:
import os
import pandas as pd

def merge_excel_files(directory):
    # getting list of Excel files in the directory
    excel_files = [file for file in os.listdir(directory) if file.endswith('.xlsx')]

    biomarker_dataset = pd.DataFrame()

    # looping through each Excel file
    for file in excel_files:
        # reading Excel file into a DataFrame
        df = pd.read_excel(os.path.join(directory, file))
        
        # extracting filename without extension as suffix
        suffix = os.path.splitext(file)[0]
        
        # merging the current DataFrame with the merged DataFrame
        if biomarker_dataset.empty:
            biomarker_dataset = df
        else:
            biomarker_dataset = pd.merge(biomarker_dataset, df, on='PTID', how='outer', suffixes=('', '_' + suffix))

    return biomarker_dataset

# directory containing Excel files
directory = r"D:\Biomarkers"
# calling the function to merge Excel files
biomarker_dataset = merge_excel_files(directory)

biomarker_dataset.rename(columns={'PTID': 'Patient ID'}, inplace=True)
print(biomarker_dataset)

In [ ]:
# counting NaNs in each column
nan_counts = biomarker_dataset.isna().sum()

# displaying the counts
print("NaN counts per column:")
print(nan_counts)
print(biomarker_dataset.columns)

In [ ]:
biomarker_dataset.to_csv('BiomarkerDataCSV', index=False)

### Genetic Dataset 
#### (contains records for 272 patients in ADNI1)

Initialises a list named `genes_list` containing the names of genes known or thought associated with Alzheimer's disease, taken from AlzPedia (https://www.alzforum.org/alzpedia). These are crucial for genetic analysis and are expected to be found in the genetic data files.

The code then reads in all VCF (Variant Call Format) files located in a specified directory. It iterates over each VCF file and extracts relevant genetic information, such as the Patient ID and associated genes related to AD.

For each VCF file, it opens the file and reads through each line. It checks if the line is not a header or comment, splits the line by tabs, and extracts gene information from the INFO field, focusing on the GI (Gene ID) information. If the gene information is available and the gene is listed in the `genes_list`, it appends the Patient ID and associated gene to the `ad_genes_data list`.

After collecting the genetic data, it creates a temporary DataFrame named df with columns "Patient ID" and "Associated Gene" to store the extracted data. It pivots the DataFrame to count the occurrence of each gene for each patient, resulting in a summary table. The index is reset to flatten multi-level columns, and the pivot table is converted back to a DataFrame named `genetic_dataset`.

The first 10 entries of the `genetic_dataset` DataFrame are printed, and the entire DataFrame is saved to a CSV file named "GeneticDataCSV".

In [ ]:
# setting the list of genes associated with AD
genes_list = [
    "ABCA7", "ADAM10", "APOE", "APP", 
    "CLU", "LRRK2", "PICALM", "SORL1",
    "TREM2", "CD2AP"
    ]

# reading in all VCF files in specified directory
directory = "D:/GeneticDataVCFs" 
vcf_files = [os.path.join(directory, file) for file in os.listdir(directory) if file.endswith('.vcf')]

ad_genes_data = []

# iterating over each VCF file
for filename in vcf_files:
    patientID = os.path.splitext(os.path.basename(filename))[0].split('_SNPs')[0]  
    with open(filename, 'r') as vcf_file:
        for line in vcf_file:
            
            # checking if line is not a header or comment
            if not line.startswith('##'):
                # splitting line using tabs as a delimiter
                parts = line.strip().split('\t')
                
                # extracting gene information from INFO field
                gi_info = [x.split('=')[1] for x in parts[7].split(';') if x.startswith('GI=')]
                
                # checking if gene information is available, if yes then extracting gene name
                if gi_info:
                    gi = gi_info[0]
                    
                     # if the gene is in the list of genes associated with AD, then append to dataframe
                    if gi in genes_list:
                        ad_genes_data.append([patientID, gi])

# initialising a new temporary DataFrame
df = pd.DataFrame(ad_genes_data, columns=['Patient ID', 'Associated Gene'])

# pivoting DataFrame to count the occurrence of each gene for each patient
pivot_table = df.pivot_table(index='Patient ID', columns='Associated Gene', aggfunc='size', fill_value=0)

# resetting index to flatten multi-level columns
pivot_table.reset_index(inplace=True)

# converting pivot table back to DataFrame
genetic_dataset = pd.DataFrame(pivot_table)

In [ ]:
# displaying first 10 files in the resultant genetic data DataFrame and saving to .CSV
print(genetic_dataset.head(10))
genetic_dataset.to_csv('GeneticDataCSV', index=False)

In [ ]:
genetic_dataset_read = pd.read_csv("C:/Users/kishe/Documents/Year 3 Jupyter/GeneticDataCSV")
genetic_dataset_read.shape[0]

### Clinical Data
#### (contains records for all 818 patients in ADNI1)

For the clinical dataset, the Excel spreadsheet is read into a Pandas DataFrame and prints the number of rows in the dataset along with the first few rows to provide an overview. A list named `columns_to_keep` is defined, containing the names of columns which are relevant for analysis. 

The code then renames the 'PTID' column to 'Patient ID' for clarity and keeps the columns specified in columns_to_keep. After filtering, it prints the first ten rows of the modified dataset. 

Finally, the processed clinical dataset is exported to a CSV file named 'ClinicalDatasetCSV', excluding the index column. This code helps data preprocessing by selecting necessary columns and organizing them into a CSV file for further analysis.

In [ ]:
clinical_dataset = pd.read_excel("D:/ClinicalDataCSV.xlsx")
print(clinical_dataset.shape[0])
print(clinical_dataset.head())

In [ ]:
columns_to_keep = [
    'PTID', 'AGE', 'PTGENDER', 'PTEDUCAT', 'PTETHCAT', 'PTRACCAT',
    'PTMARRY', 'APOE4', 'FDG', 'ABETA', 'TAU', 'PTAU',
    'CDRSB', 'ADAS11', 'ADAS13', 'ADASQ4', 'MMSE', 'RAVLT_immediate',
    'RAVLT_learning', 'RAVLT_forgetting', 'RAVLT_perc_forgetting',
    'LDELTOTAL', 'DIGITSCOR', 'TRABSCOR', 'FAQ', 'Ventricles', 'Hippocampus', 'WholeBrain',
    'Entorhinal', 'Fusiform', 'MidTemp', 'ICV', 'DX', 'mPACCdigit',
    'mPACCtrailsB'
]

# keeping only the columns in columns_to_keep
clinical_dataset = clinical_dataset.filter(columns_to_keep)

# renaming columns for consistency
clinical_dataset.rename(columns={'PTID': 'Patient ID', 'DX': 'Diagnosis'}, inplace=True)

print(clinical_dataset.head(10))

In [ ]:
print(clinical_dataset.columns)

In [ ]:
clinical_dataset.to_csv('ClinicalDatasetCSV', index=False)

### Combining All Datasets

This merges three DataFrames, `clinical_dataset`, `genetic_dataset` and `biomarker_dataset`, based on their shared 'Patient ID' column, Using the `pandas merge` function with an 'outer' join type. Suffixes `_Biomarker`, `_Clinical` and `_Genetic` are appended to the column names to distinguish them in the merged DataFrame.

The code checks if each dataset has a 'Patient ID' column, essential for merging the datasets based on unique patient identifiers. If the 'Patient ID' column exists in a dataset, a corresponding message is printed indicating its presence.

The next step involves merging the datasets based on the 'Patient ID' column using an outer join. The resulting merged dataset, named `merged_dataset`, combines information from the clinical, biomarker, and genetic datasets.

Lastly, the code removes rows with NaN values in the 'PTEDUCAT' and 'PROTEIN' column from the resulting merged DataFrame, collating a list of patients with entries in the clinical modality, ensuring they have a diagnosis. Only a small percentage of patients in the ADNI1 study have corresponding genetic and biomarker data, and missing values will be imputed later. Finally, the resulting `merged_dataset` is saved to a CSV file named 'MergedDataSetCSV' using the `to_csv()` function.

In [ ]:
# reading each dataset from their CSV files
clinical_dataset = pd.read_csv(r"C:\Users\kishe\Documents\Year 3 Jupyter\ClinicalDatasetCSV")
# replacing 'Dementia' with 'AD' in the 'Diagnosis' column
clinical_dataset['Diagnosis'] = clinical_dataset['Diagnosis'].replace('Dementia', 'AD')

genetic_dataset = pd.read_csv(r"C:\Users\kishe\Documents\Year 3 Jupyter\GeneticDataCSV")

biomarker_dataset = pd.read_pickle(r"C:\Users\kishe\Documents\Year 3 Jupyter\BiomarkerProcessedDataCSV")

print(clinical_dataset.shape)
print(clinical_dataset.groupby('Diagnosis').size())
print(genetic_dataset.shape)
print(biomarker_dataset.shape)

In [ ]:
# checking if 'Patient ID' column exists in each of the CSV datasets
if 'Patient ID' in clinical_dataset.columns:
    print("clinical_dataset has a 'Patient ID' column")
else:
    print("clinical_dataset does not have a 'Patient ID' column")

if 'Patient ID' in biomarker_dataset.columns:
    print("biomarker_dataset has a 'Patient ID' column")
else:
    print("biomarker_dataset does not have a 'Patient ID' column")

if 'Patient ID' in genetic_dataset.columns:
    print("genetic_dataset has a 'Patient ID' column")
else:
    print("genetic_dataset does not have a 'Patient ID' column")


In [ ]:
# merging DataFrames based on 'Patient ID' column
merged_dataset = pd.merge(clinical_dataset, biomarker_dataset, on='Patient ID', how='outer', suffixes=('_Clinical', '_Biomarker'))
merged_dataset = pd.merge(merged_dataset, genetic_dataset, on='Patient ID', how='outer', suffixes=('', '_Genetic'))

# dropping rows with NaN values in clinical columns so that only patients with both clinical and biomarker are kept
merged_dataset.dropna(subset=['PTEDUCAT', 'CTRED'], inplace=True)

# displaying the resulting DataFrame
print(merged_dataset.shape[0])
print(merged_dataset.head(10))
print(merged_dataset.columns)

In [ ]:
merged_dataset.to_csv('MergedDataSetCSV', index=False)
merged_dataset.to_pickle('MergedDataSetPKL')

In [ ]:
merged_dataset = pd.read_pickle(r"C:\Users\kishe\Documents\Year 3 Jupyter\MergedDataSetPKL")
merged_dataset.groupby('Diagnosis').size()

### Genetic Data - - Imputing Missing Values

To impute the missing genetic data in some patients, the `ExtraTreesRegressor` from `scikit-learn` is imported to address missing values in specific columns. It starts by defining the columns needing imputation. 

Then, it initialises the `IterativeImputer` with selected hyperparameters. Subsequently, missing values in the dataset are imputed using the `fit` and `transform()` methods, respectively, with the `fit` function being called on the combined clinical and genetic dataset taken from ADNI2, 3 and 4. The resulting imputed datasets are rounded to integers. As a result, the patients in ADNI1 that were missing gene counts now have an estimated set of values based on other patient data with whom they share similar clinical data values.

The imputed values are floored to 0 if negative and then rounded for completeness. Finally, the first 10 rows of the imputed dataset are printed for validation. This process ensures that missing values are appropriately handled, enhancing the datasets' integrity for further analysis or modeling.

**Note:** For the purposes of this project, the genetic data has been imputed before splitting the datasets, which does not follow an orthodox preprocessing pipeline. This is due to a few reasons:

- Imputing missing values before splitting ensures that no information is lost during the data splitting process, particularly important in this case as only 272 out of 818 patients in ADNI1 have corresponding genetic data.
- By imputing missing values before splitting, the distribution and characteristics of the data remain consistent across the training, validation, and test sets. This can lead to more reliable model performance evaluation, as the model learns from data that is more representative of the overall dataset.
- Imputing missing values before splitting allows the model to learn from a more complete dataset, which can potentially lead to better model performance.

In [ ]:
# columns to impute
columns_to_impute = ['ABCA7', 'ADAM10', 'APOE', 'APP', 'CD2AP', 'CLU', 'LRRK2', 'PICALM', 'SORL1', 'TREM2']

# finding indices of missing values in each dataset
missing_indices_train = np.where(np.isnan(merged_dataset[columns_to_impute]))

temp_full_clinical_dataset = pd.read_csv(r"D:\ADNIMERGE_BL.csv")

# replacing 'Dementia' with 'AD' in the 'Diagnosis' column
temp_full_clinical_dataset['Diagnosis'] = temp_full_clinical_dataset['Diagnosis'].replace('Dementia', 'AD')
temp_full_clinical_dataset = temp_full_clinical_dataset.dropna(subset=['Diagnosis'])

# reading the full ADNI1/2/3/4 genetic dataset in and merging with the full ADNI1/2/3/4 clinical dataset on 'Patient ID'
temp_full_genetic_dataset = pd.read_csv(r"C:\Users\kishe\Documents\Year 3 Jupyter\GeneticDataCSV")

combined_dataset = pd.merge(temp_full_clinical_dataset, temp_full_genetic_dataset, on='Patient ID', how='inner')
combined_dataset = combined_dataset[combined_dataset.columns.intersection(merged_dataset.columns)]
temp_merged_dataset = merged_dataset[combined_dataset.columns.intersection(merged_dataset.columns)]

# columns to drop from combined and temp merged datasets before imputing as they are non-numerical
columns_to_drop = ['Patient ID', 'Diagnosis', 'AGE', 'PTGENDER', 'PTEDUCAT', 'PTETHCAT', 'PTRACCAT', 'PTMARRY', 'APOE4', 'ABETA', 'TAU', 'PTAU']
combined_dataset.drop(columns=columns_to_drop, inplace=True)
temp_merged_dataset.drop(columns=columns_to_drop, inplace=True)

print(combined_dataset.shape[0])
print(combined_dataset.columns)
print(temp_merged_dataset.shape[0])
print(temp_merged_dataset.columns)

In [ ]:
# columns to impute
columns_to_impute = ['ABCA7', 'ADAM10', 'APOE', 'APP', 'CD2AP', 'CLU', 'LRRK2', 'PICALM', 'SORL1', 'TREM2']

# initialising the estimator to use for imputation
estimator = ExtraTreesRegressor(n_estimators=500, min_samples_split=100, min_samples_leaf=60, random_state=42)

# initialising IterativeImputer and fitting on combined dataset created above for accurate imputed values
imputer = IterativeImputer(estimator=estimator, max_iter=25, random_state=42, imputation_order='ascending')
imputer.fit(combined_dataset)
# imputing missing values for temp_merged_dataset
temp_merged_dataset_imputed = imputer.transform(temp_merged_dataset)

# flooring negative values to 0 as negative gene counts are not possible
temp_merged_dataset_imputed[temp_merged_dataset_imputed < 0] = 0
# rounding gene counts as fractional values are not possible
temp_merged_dataset_imputed = np.round(temp_merged_dataset_imputed)

# converting temp_merged_dataset_imputed to a DataFrame
temp_merged_dataset_imputed_df = pd.DataFrame(temp_merged_dataset_imputed, columns=temp_merged_dataset.columns)

# updating the original merged_dataset with the imputed values
merged_dataset[columns_to_impute] = temp_merged_dataset_imputed[:, :len(columns_to_impute)]

print(merged_dataset[columns_to_impute].head(10))

### Biomarker Data - Imputing Missing Values
Follows the same process as the Genetic Imputer, but applied to the Biomarker column data instead.

In [ ]:
# columns to impute
columns_to_impute = ['CTWHITE', 'CTRED', 'PROTEIN', 'GLUCOSE', 'CTWHITE_M12',
       'CTRED_M12', 'PROTEIN_M12', 'GLUCOSE_M12', 'CTWHITE_M24', 'CTRED_M24',
       'PROTEIN_M24', 'GLUCOSE_M24', 'CTWHITE_M36', 'CTRED_M36', 'PROTEIN_M36',
       'GLUCOSE_M36', 'CTWHITE_M48', 'CTRED_M48', 'PROTEIN_M48',
       'GLUCOSE_M48']

# finding indices of missing values in each dataset
missing_indices_train = np.where(np.isnan(merged_dataset[columns_to_impute]))

temp_full_clinical_dataset = pd.read_csv("D:/ADNIMERGE_BL.csv")
temp_full_clinical_dataset = temp_full_clinical_dataset.dropna(subset=['Diagnosis'])

temp_full_biomarker_dataset = pd.read_csv("C:/Users/kishe/Documents/Year 3 Jupyter/BiomarkerImputerCSV")

# dictionary to map old column names to new column names
column_mapping = {
    'CTWHITE_M12Imp': 'CTWHITE_M12',
    'CTRED_M12Imp': 'CTRED_M12',
    'PROTEIN_M12Imp': 'PROTEIN_M12',
    'GLUCOSE_M12Imp': 'GLUCOSE_M12',
    'CTWHITE_M24Imp': 'CTWHITE_M24',
    'CTRED_M24Imp': 'CTRED_M24',
    'PROTEIN_M24Imp': 'PROTEIN_M24',
    'GLUCOSE_M24Imp': 'GLUCOSE_M24',
    'CTWHITE_M36Imp': 'CTWHITE_M36',
    'CTRED_M36Imp': 'CTRED_M36',
    'PROTEIN_M36Imp': 'PROTEIN_M36',
    'GLUCOSE_M36Imp': 'GLUCOSE_M36',
    'CTWHITE_M48Imp': 'CTWHITE_M48',
    'CTRED_M48Imp': 'CTRED_M48',
    'PROTEIN_M48Imp': 'PROTEIN_M48',
    'GLUCOSE_M48Imp': 'GLUCOSE_M48'
}

# renaming columns in the DataFrame and dropping Patient ID
temp_full_biomarker_dataset.rename(columns=column_mapping, inplace=True)

combined_dataset = pd.merge(temp_full_clinical_dataset, temp_full_biomarker_dataset, on='Patient ID', how='inner')
combined_dataset = combined_dataset[combined_dataset.columns.intersection(merged_dataset.columns)]
temp_merged_dataset = merged_dataset[combined_dataset.columns.intersection(merged_dataset.columns)]

# columns to drop from combined and temp merged datasets before imputing as they are non-numerical
columns_to_drop = ['Patient ID', 'Diagnosis', 'AGE', 'PTGENDER', 'PTEDUCAT', 'PTETHCAT', 'PTRACCAT', 'PTMARRY', 'APOE4', 'ABETA', 'TAU', 'PTAU']
combined_dataset.drop(columns=columns_to_drop, inplace=True)
temp_merged_dataset.drop(columns=columns_to_drop, inplace=True)

print(combined_dataset.shape[0])
print(combined_dataset.columns)
print(temp_merged_dataset.shape[0])
print(temp_merged_dataset.columns)


In [ ]:
# columns to impute
columns_to_impute = ['CTWHITE', 'CTRED', 'PROTEIN', 'GLUCOSE', 'CTWHITE_M12',
       'CTRED_M12', 'PROTEIN_M12', 'GLUCOSE_M12', 'CTWHITE_M24', 'CTRED_M24',
       'PROTEIN_M24', 'GLUCOSE_M24', 'CTWHITE_M36', 'CTRED_M36', 'PROTEIN_M36',
       'GLUCOSE_M36', 'CTWHITE_M48', 'CTRED_M48', 'PROTEIN_M48',
       'GLUCOSE_M48']

# initialising the estimator to use for imputation
estimator = ExtraTreesRegressor(n_estimators=60, min_samples_split=15, min_samples_leaf=5, random_state=42)

# initialising IterativeImputer and fitting on combined dataset created above for accurate imputed values
imputer = IterativeImputer(estimator=estimator, random_state=42)

#imputer.fit(temp_full_biomarker_dataset)
imputer.fit(combined_dataset)

# imputing missing values for temp_merged_dataset
temp_merged_dataset_imputed = imputer.transform(temp_merged_dataset)

# flooring negative values to 0 as negative gene counts are not possible
temp_merged_dataset_imputed[temp_merged_dataset_imputed < 0] = 0
# rounding gene counts as fractional values are not possible
temp_merged_dataset_imputed = np.round(temp_merged_dataset_imputed)

# converting temp_merged_dataset_imputed to a DataFrame
temp_merged_dataset_imputed_df = pd.DataFrame(temp_merged_dataset_imputed, columns=temp_merged_dataset.columns)

# updating the original merged_dataset with the imputed values
merged_dataset[columns_to_impute] = temp_merged_dataset_imputed_df[columns_to_impute]

print(merged_dataset[columns_to_impute].head(10))

### Splitting The Datasets

After being combined, the merged dataset is split into features and labels, with labels derived from the 'Diagnosis' column. Utilising the `train_test_split` function from sklearn, each dataset is split into training, validation, and test sets, maintaining a 70:15:15 ratio. Finally, the code confirms the integrity of the splits by printing the shapes of the resulting datasets, ensuring accurate distribution of samples and features across sets.

In [ ]:
diagnosis_counts = merged_dataset.groupby('Diagnosis').size()
print(diagnosis_counts)

In [ ]:
X_merged = merged_dataset
# splitting merged dataset into training, validation, and test sets
X_train_merged, X_temp_merged = train_test_split(X_merged, test_size=0.3, random_state=42)
X_val_merged, X_test_merged = train_test_split(X_temp_merged, test_size=0.5, random_state=42)

In [ ]:
# printing the shapes of the resulting datasets
print("Merged Dataset Shapes:")
print("Train:", X_train_merged.shape) #, y_train_merged.shape)
print("Validation:", X_val_merged.shape) #, y_val_merged.shape)
print("Test:", X_test_merged.shape) #, y_test_merged.shape)

## Cleaning, Imputing & Standardising Data

### Clinical Data - Cleaning, Encoding & Normalising Clinical Values

The following code performs several data preprocessing tasks across the clinical data. Firstly, a function named `replace_inequality()` is defined to replace inequality symbols in specific columns with appropriate numeric values. Next, a list of columns to process is specified. Then, for each dataset in the list, the following steps are executed:

1. Reset the dataset index.
2. Iterate over each column in the `columns_to_process list.
3. Replace inequality symbols with numeric values using the `replace_inequality()` function and handle non-numeric values.
4. Convert the column data to numeric type, coercing errors to 'NaN' for non-numeric values.
5. Interpolate missing values only in numeric columns.

After this preprocessing, standardisation is applied to specified columns across all datasets. `StandardScaler` from `scikit-learn` is used. The specified columns are standardized using the `fit_transform()` method.

Finally, the code checks for any NaN values in the merged training dataset by iterating over each row and column. If a NaN value is found, the row data containing the NaN value is printed for inspection.

In [ ]:
# defining a function to replace inequality symbols with numeric values
def replace_inequality(value, symbol):
    if symbol == ">":
        return float(value[1:]) + 1  # extracting numeric part and converting to float before adding 1
    elif symbol == "<":
        return float(value[1:]) - 1  # extracting numeric part and converting to float before subtracting 1
    else:
        return value

columns_to_process = ['FDG', 'ABETA', 'TAU', 'PTAU', 'LDELTOTAL', 'DIGITSCOR', 'TRABSCOR', 'FAQ',
                      'Ventricles', 'Hippocampus', 'WholeBrain', 'Entorhinal', 'Fusiform', 'MidTemp', 'ICV', 'ADAS11',
                      'ADAS13', 'RAVLT_immediate', 'RAVLT_learning', 'RAVLT_forgetting', 'RAVLT_perc_forgetting']  

for dataset in [X_train_merged, X_val_merged, X_test_merged]:
    # resetting index of DataFrame
    dataset.reset_index(drop=True, inplace=True)

    for column in columns_to_process:
        # replacing inequality symbols with numeric values
        dataset[column] = dataset.apply(lambda row: replace_inequality(row[column], row[column][0]) if isinstance(row[column], str) and row[column][0] in ['<', '>'] else row[column], axis=1)
        
        # converting to numeric with errors set to 'coerce' to handle non-numeric values
        dataset[column] = pd.to_numeric(dataset[column], errors="coerce")

        # interpolating only if the column is numeric
        if dataset[column].dtype in ['int64', 'float64']:
            dataset[column] = dataset[column].interpolate(axis=0, limit_direction="both")

print(X_train_merged.head(10))

In [ ]:
# columns to standardise
columns_to_standardise = ['Ventricles', 'Hippocampus', 'WholeBrain', 'Entorhinal', 'Fusiform', 'MidTemp', 'ICV']

# initialise StandardScaler
scaler = MinMaxScaler()

# applying standardisation to the specified columns in each dataset
for dataset in [X_train_merged, X_val_merged, X_test_merged]:
    dataset.reset_index(drop=True, inplace=True)
    dataset[columns_to_standardise] = scaler.fit_transform(dataset[columns_to_standardise])

print(X_train_merged.head(10))

Normalising numerical columns is important because it helps to scale the features to a similar range, preventing features with larger magnitudes from dominating those with smaller magnitudes during the training of machine learning models. This can lead to more stable and faster convergence during training.

In [ ]:
row = X_train_merged.iloc[0]
print(row)

In [ ]:
# initialising OneHotEncoder and defining categorical columns
encoder = OneHotEncoder(sparse_output=False, drop='first')
categorical_columns = ['PTGENDER', 'PTETHCAT', 'PTRACCAT', 'PTMARRY', 'APOE4']

# fitting and transforming the categorical columns in each dataset
for dataset in [X_train_merged, X_val_merged, X_test_merged]:
    dataset.reset_index(drop=True, inplace=True)
    
    # encoding categorical columns
    encoded_data = encoder.fit_transform(dataset[categorical_columns])
    
    # assigning encoded values back to original dataset
    for i, col in enumerate(encoder.get_feature_names_out(categorical_columns)):
        dataset[col] = encoded_data[:, i]

    # dropping original categorical columns
    dataset.drop(categorical_columns, axis=1, inplace=True)

print("Encoding completed for all datasets.")
print(X_train_merged.head(10))

### Checking Class Imbalances & Fixing

Importing the RandomOverSampler class from the `imbalanced-learn` library, and then checking the class distribution of target variables for training, validation, and testing datasets. 

Any class imbalances are addressed using random over-sampling, a technique to balance classes by duplicating instances from minority classes. The `fit_resample()` method applies this technique to rebalance the datasets, and the balanced class distribution is printed to confirm successful rebalancing.

In [ ]:
# extracting the 'Diagnosis' column from the X datasets
diagnosis_train = X_train_merged['Diagnosis']
diagnosis_val = X_val_merged['Diagnosis']
diagnosis_test = X_test_merged['Diagnosis']

# checking class distribution
y_train_distribution = diagnosis_train.value_counts()
y_val_distribution = diagnosis_val.value_counts()
y_test_distribution = diagnosis_test.value_counts()

print("Class distribution of Diagnosis in X_train_merged:", y_train_distribution)
print("Class distribution of Diagnosis in X_val_merged:", y_val_distribution)
print("Class distribution of Diagnosis in X_test_merged:", y_test_distribution)

In [ ]:
# addressing class imbalances with random over-sampling
ros = RandomOverSampler(random_state=42)
X_train_merged, y_train = ros.fit_resample(X_train_merged, diagnosis_train)
X_val_merged, y_val = ros.fit_resample(X_val_merged, diagnosis_val)
X_test_merged, y_test = ros.fit_resample(X_test_merged, diagnosis_test)

# printing balanced class distribution
print("Balanced class distribution, X_train_merged:")
print(pd.Series(X_train_merged['Diagnosis']).value_counts())
print("Balanced class distribution, X_val_merged:")
print(pd.Series(X_val_merged['Diagnosis']).value_counts())
print("Balanced class distribution, X_test_merged:")
print(pd.Series(X_test_merged['Diagnosis']).value_counts())

In [ ]:
# counting NaNs in each column and displaying for each X set to check all values are present/imputed
train_nan_counts = X_train_merged.isna().sum()
print("NaN counts per column:")
print(train_nan_counts)

val_nan_counts = X_val_merged.isna().sum()
print("NaN counts per column for validation set:")
print(val_nan_counts)

test_nan_counts = X_test_merged.isna().sum()
print("\nNaN counts per column for test set:")
print(test_nan_counts)

### Saving All Datasets to .PKL

In [ ]:
X_train_merged.to_pickle("X_train_merged.pkl")
X_val_merged.to_pickle("X_val_merged.pkl")
X_test_merged.to_pickle("X_test_merged.pkl")

### Plotting Graphs for Data Analysis

In [ ]:
# plotting a bar plot of gene counts
plt.figure(figsize=(12, 6))
genetic_dataset.drop(['Patient ID'], axis=1).sum().plot(kind='bar', color='skyblue')
plt.title('Count of Genes Associated with Alzheimer\'s Disease')
plt.xlabel('Gene')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# plotting a histogram of patient ages
plt.figure(figsize=(10, 6))
clinical_dataset['AGE'].plot(kind='hist', bins=20, color='orange', edgecolor='black')
plt.title('Distribution of Patient Ages')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# plotting a scatter plot of MRI features by diagnosis
plt.figure(figsize=(10, 5.5))
sns.scatterplot(data=clinical_dataset, x='Diagnosis', y='AGE', hue='PTGENDER', palette='viridis')
plt.title('Scatter Plot of Age, Sex and Diagnosis')
plt.xlabel('Diagnosis')
plt.ylabel('Age')
plt.legend(title='Diagnosis')
plt.tight_layout()
plt.show()